# E08 Bike Sharing：时间验证与可用滞后特征

> Issue #9 / Epic #6 / 路线图 E08

## 你将学到
1. 必须按时间切分，禁止随机打乱
2. 滞后特征只能用预测时刻已知信息
3. 常用 RMSLE 作为指标

## 数据
- 本地 `data/bike_sharing_hour.csv`
- 回退：OpenML / 合成时间序列 DEMO


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载数据


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/bike_sharing_hour.csv')
DEMO = False
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml('Bike_Sharing_Demand', version=1, as_frame=True, parser='auto')
        df = raw.frame.copy()
        print('OpenML', df.shape)
    except Exception:
        DEMO = True
        print('使用合成时间序列 DEMO')
        n = 2000 if QUICK else 5000
        rng = np.random.RandomState(RANDOM_STATE)
        idx = pd.date_range('2012-01-01', periods=n, freq='h')
        df = pd.DataFrame({
            'datetime': idx,
            'temp': 20 + 10 * np.sin(np.arange(n) / 24) + rng.normal(0, 2, n),
            'hum': 60 + rng.normal(0, 10, n),
            'windspeed': np.abs(rng.normal(10, 3, n)),
            'season': ((idx.month % 12) // 3 + 1),
            'holiday': 0,
            'workingday': (idx.dayofweek < 5).astype(int),
            'cnt': None,
        })
        df['cnt'] = (
            100 + 40 * np.sin(np.arange(n) / 24) + 20 * df['workingday'] + rng.normal(0, 15, n)
        ).clip(0, None)

# 统一目标列
if 'cnt' not in df.columns:
    for c in ['count', 'Count', 'total']:
        if c in df.columns:
            df = df.rename(columns={c: 'cnt'})
            break
print(df.head())
print('DEMO', DEMO)


## 2. 时间特征与排序


In [ ]:
work = df.copy()
# 解析时间
time_col = None
for c in ['dteday', 'datetime', 'date', 'timestamp']:
    if c in work.columns:
        time_col = c
        break
if time_col:
    work[time_col] = pd.to_datetime(work[time_col])
    work = work.sort_values(time_col).reset_index(drop=True)
    work['hour'] = work[time_col].dt.hour if hasattr(work[time_col].dt, 'hour') else 0
    work['dow'] = work[time_col].dt.dayofweek
else:
    work = work.reset_index(drop=True)
    work['hour'] = work['hr'] if 'hr' in work.columns else 0
    work['dow'] = work['weekday'] if 'weekday' in work.columns else 0

print(work.shape)


## 3. 安全滞后特征（只用过去）


In [ ]:
# 滞后 1/24 小时的 cnt；验证集构造时注意 shift 不穿越未来
work['cnt_lag1'] = work['cnt'].shift(1)
work['cnt_lag24'] = work['cnt'].shift(24)
work['cnt_roll24'] = work['cnt'].shift(1).rolling(24).mean()
work = work.dropna().reset_index(drop=True)

split = int(len(work) * 0.8)
train, valid = work.iloc[:split], work.iloc[split:]
drop_cols = [c for c in ['cnt', time_col, 'casual', 'registered', 'instant', 'dteday', 'datetime'] if c and c in work.columns]
feature_cols = [c for c in work.columns if c not in drop_cols]
# 只用数值
Xtr = train[feature_cols].select_dtypes(include=[np.number])
Xva = valid[feature_cols].select_dtypes(include=[np.number])
ytr, yva = train['cnt'], valid['cnt']
print('features', list(Xtr.columns))
print('train', Xtr.shape, 'valid', Xva.shape)


## 4. 基线 + RMSLE


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import mean_squared_log_error, mean_absolute_error

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr.fillna(0), ytr, Xva.fillna(0), yva, task_type='regression')
comparison = baseline.compare_models(results, top_n=5)
display(comparison)
best = comparison.iloc[0]['Model']
pred = np.clip(np.asarray(results[best]['predictions']), 0, None)
print('MAE', mean_absolute_error(yva, pred))
print('RMSLE', mean_squared_log_error(yva, pred, squared=False))


## 5. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/08_bike_sharing'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred)), 'cnt': yva.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 7. 检查清单
- [x] 时间切分
- [x] 滞后特征只用过去
- [x] RMSLE
- [ ] 天气/节假日交互
- [ ] 禁止把 casual+registered 当特征（泄漏）
